<a href="https://colab.research.google.com/github/iknoest/fair-feeder/blob/claude%2Fcat-feeder-dataset-v13-yQuRz/smoketest.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fair Feeder V13 — Smoke Test & Feeding Analysis

Run YOLOv11 inference on test images and videos from the Tapo IR camera.

**What this notebook does:**
1. Validates model accuracy on the test split (mAP, per-class AP50)
2. Benchmarks inference speed
3. Runs inference on images/videos from Google Drive
4. Tracks feeding behaviour: kibble count, cat-at-bowl, Dan's hand episodes
5. Extracts timestamps from Tapo's burned-in OSD via OCR
6. Sends a structured summary + snapshots to Discord

In [ ]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# 2. Install dependencies
!pip install -q ultralytics roboflow easyocr tqdm requests
!nvidia-smi

In [ ]:
from ultralytics import YOLO
import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from pathlib import Path
from IPython.display import display, Video, Image as IPImage
from collections import defaultdict
from PIL import Image
import os, glob, time, re, json, requests
from tqdm.notebook import tqdm
import easyocr

In [ ]:
# Download dataset for validation metrics
ROBOFLOW_API_KEY = "YOUR_API_KEY"  # <-- paste your Roboflow key here

from roboflow import Roboflow
rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace("test-7vyqo").project("ir-kibble")
version = project.version(13)
dataset = version.download("yolov8")
DATASET_PATH = dataset.location
dataset_path = Path(DATASET_PATH)
data_yaml = str(dataset_path / "data.yaml")
print(f"\nDataset: {DATASET_PATH}")

In [ ]:
# Fix Labels: Polygon → Bounding Box
# Roboflow exports mixed polygon+bbox labels. Without this fix, YOLO drops
# segment annotations during val(), giving wrong metrics. Same fix as
# Step 3 in the training notebook.

import shutil

def is_polygon_line(line):
    return len(line.strip().split()) > 5

def polygon_to_bbox(line):
    parts = line.strip().split()
    class_id = parts[0]
    coords = [float(v) for v in parts[1:]]
    xs = coords[0::2]
    ys = coords[1::2]
    x_min, x_max = min(xs), max(xs)
    y_min, y_max = min(ys), max(ys)
    x_center = (x_min + x_max) / 2
    y_center = (y_min + y_max) / 2
    width = x_max - x_min
    height = y_max - y_min
    return f"{class_id} {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}"

total_converted = 0
for split in ["train", "valid", "test"]:
    label_dir = dataset_path / split / "labels"
    if not label_dir.is_dir():
        print(f"[{split}] No labels directory — skipping")
        continue

    # Backup originals
    backup_dir = dataset_path / split / "labels_backup_polygon"
    if not backup_dir.exists():
        shutil.copytree(label_dir, backup_dir)

    # Convert
    split_converted = 0
    for f in sorted(label_dir.glob("*.txt")):
        lines = f.read_text().splitlines()
        new_lines = []
        for line in lines:
            if not line.strip():
                continue
            if is_polygon_line(line):
                new_lines.append(polygon_to_bbox(line))
                split_converted += 1
            else:
                new_lines.append(line.strip())
        f.write_text("\n".join(new_lines) + "\n" if new_lines else "")

    # Verify
    remaining = sum(
        1 for f in label_dir.glob("*.txt")
        for line in f.read_text().splitlines()
        if line.strip() and is_polygon_line(line)
    )
    status = "PASS" if remaining == 0 else f"FAIL ({remaining} polygons left)"
    print(f"[{split}] Converted {split_converted} polygon lines — Verification: {status}")
    total_converted += split_converted

print(f"\nTotal polygon lines converted: {total_converted}")

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────
MODEL_PATH = '/content/drive/MyDrive/Fun Project/Cat monitor/model/fair_feeder_v13_yolov11s.pt'
SOURCE_DIR = '/content/drive/MyDrive/Fun Project/Cat monitor/Test_postmodel'
OUTPUT_DIR = '/content/drive/MyDrive/Fun Project/Cat monitor/Test_postmodel_output'

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Discord ───────────────────────────────────────────────────────
DISCORD_WEBHOOK_URL = ""  # <-- paste your Discord webhook URL here

# ── Class colours (RGB for display, BGR for OpenCV) ───────────
CLASS_COLORS_RGB = {
    "Dan":      (0,   255, 206),  # #00FFCE  teal
    "Bowl":     (134,  34, 255),  # #8622FF  purple
    "Dan_hand": (0,   16,  255),  # #0010FF  blue
    "Kibble":   (252, 244,   0),  # #FCF400  yellow
    "Sanbo":    (255, 128,   0),  # #FF8000  orange
}
CLASS_ORDER = ["Dan", "Sanbo", "Dan_hand", "Bowl", "Kibble"]

# ── Dark background colours per class (for summary table) ──────
CLASS_ROW_BG = {
    "Dan":      "#003D2F",
    "Sanbo":    "#3D2000",
    "Dan_hand": "#00073D",
    "Kibble":   "#3D3A00",
    "Bowl":     "#250050",
}

# ── Inference settings ───────────────────────────────────────────
CONF_THRESHOLD      = 0.45
IOU_THRESHOLD       = 0.20
IMGSZ               = 1280   # single int — YOLO auto-pads to stride-32
GRAY_DIFF_THRESHOLD = 15     # mean channel diff below this → IR image

# ── Behavioural thresholds ───────────────────────────────────────
OVERLAP_IOU_THRESHOLD = 0.05   # bbox overlap to count "cat at bowl"
EATING_KIBBLE_DROP    = 2      # min kibble count decrease to confirm eating
DAN_HAND_GAP_FRAMES  = 15     # frames without Dan_hand = new episode

# ── OCR ───────────────────────────────────────────────────────────
TIMESTAMP_CROP = (0, 0, 450, 45)  # (x1, y1, x2, y2) for Tapo timestamp region
OCR_EVERY_N_FRAMES = 30            # OCR every Nth frame (~1/sec at 25fps)

# ── File extensions ──────────────────────────────────────────────
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp"}
VIDEO_EXTENSIONS = {".mp4", ".avi", ".mov", ".mkv"}

print("✅ Config loaded.")
print(f"   Model  : {MODEL_PATH}")
print(f"   Source : {SOURCE_DIR}")
print(f"   Output : {OUTPUT_DIR}")

In [ ]:
model = YOLO(MODEL_PATH)

# FIX: Use model.names for correct class→index mapping
# model.names = {0: 'Bowl', 1: 'Dan', 2: 'Dan_hand', 3: 'Kibble', 4: 'Sanbo'}
CLASS_NAMES = model.names
print("✅ Model loaded.")
print(f"   Classes: {CLASS_NAMES}")

In [ ]:
# FIX: imgsz=1280 (single int) avoids "must be multiple of stride 32" warnings
# FIX: Per-class AP50 uses model.names for correct index mapping
metrics = model.val(data=data_yaml, imgsz=IMGSZ, rect=True)

print(f"mAP50:     {metrics.box.map50:.3f}")
print(f"mAP50-95:  {metrics.box.map:.3f}")
print(f"Precision: {metrics.box.mp:.3f}")
print(f"Recall:    {metrics.box.mr:.3f}")

print("\nPer-class AP50:")
for idx in sorted(CLASS_NAMES.keys()):
    name = CLASS_NAMES[idx]
    print(f"  {name:10s} {metrics.box.ap50[idx]:.3f}")

In [ ]:
# Benchmark inference speed
# FIX: imgsz=1280 (single int)
test_img = str(next(Path(DATASET_PATH, "test", "images").glob("*.jpg")))

# Warmup
for _ in range(5):
    model.predict(test_img, imgsz=IMGSZ, verbose=False)

# Benchmark
times = []
for _ in range(50):
    t0 = time.perf_counter()
    model.predict(test_img, imgsz=IMGSZ, verbose=False)
    times.append(time.perf_counter() - t0)

avg_ms = sum(times) / len(times) * 1000
fps = 1000 / avg_ms
print(f"Avg inference: {avg_ms:.1f} ms  ({fps:.1f} FPS)")

In [ ]:
# ───────────────────────────────────────────────────────────────
# OCR + Helper Functions
# ───────────────────────────────────────────────────────────────

# --- OCR reader for Tapo timestamp ---
ocr_reader = easyocr.Reader(['en'], gpu=False)
TIMESTAMP_REGEX = re.compile(r'(\d{4}[-/]\d{2}[-/]\d{2}\s+\d{2}:\d{2}:\d{2})')


def extract_timestamp(frame):
    """Crop top-left corner, OCR the Tapo burned-in timestamp."""
    x1, y1, x2, y2 = TIMESTAMP_CROP
    crop = frame[y1:y2, x1:x2]
    gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)
    _, thresh = cv2.threshold(gray, 180, 255, cv2.THRESH_BINARY)
    results = ocr_reader.readtext(thresh, detail=0)
    text = " ".join(results).strip()
    m = TIMESTAMP_REGEX.search(text)
    return m.group(1) if m else text


def classify_file(filepath):
    """Return 'image', 'video', or None based on file extension."""
    ext = Path(filepath).suffix.lower()
    if ext in IMAGE_EXTENSIONS:
        return "image"
    elif ext in VIDEO_EXTENSIONS:
        return "video"
    return None


def detect_image_type(img_bgr):
    """
    Returns 'color' or 'ir'.
    Tapo IR images are grayscale stored as BGR where R≈G≈B.
    """
    b, g, r = cv2.split(img_bgr)
    b, g, r = b.astype(int), g.astype(int), r.astype(int)
    max_diff = max(
        np.mean(np.abs(r - g)),
        np.mean(np.abs(r - b)),
        np.mean(np.abs(g - b)),
    )
    return "ir" if max_diff < GRAY_DIFF_THRESHOLD else "color"


def prepare_for_inference(img_bgr, image_type):
    """Color image → gray then back to 3-ch BGR for YOLO. IR → pass through."""
    if image_type == "color":
        gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
        return cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR)
    return img_bgr


def get_color_bgr(class_name):
    r, g, b = CLASS_COLORS_RGB.get(class_name, (255, 255, 255))
    return (b, g, r)


def bbox_iou(box_a, box_b):
    """Compute IoU between two dicts with x1,y1,x2,y2 keys."""
    xa1 = max(box_a["x1"], box_b["x1"])
    ya1 = max(box_a["y1"], box_b["y1"])
    xa2 = min(box_a["x2"], box_b["x2"])
    ya2 = min(box_a["y2"], box_b["y2"])
    inter = max(0, xa2 - xa1) * max(0, ya2 - ya1)
    area_a = (box_a["x2"] - box_a["x1"]) * (box_a["y2"] - box_a["y1"])
    area_b = (box_b["x2"] - box_b["x1"]) * (box_b["y2"] - box_b["y1"])
    union = area_a + area_b - inter
    return inter / union if union > 0 else 0.0


def parse_results(results):
    """Parse YOLO results into a list of detection dicts."""
    detections = []
    for result in results:
        if result.boxes is None:
            continue
        for box in result.boxes:
            cls_id = int(box.cls.item())
            detections.append({
                "class_name": model.names[cls_id],
                "conf": float(box.conf.item()),
                "x1": int(box.xyxy[0][0].item()),
                "y1": int(box.xyxy[0][1].item()),
                "x2": int(box.xyxy[0][2].item()),
                "y2": int(box.xyxy[0][3].item()),
            })
    return detections


def draw_boxes(img_bgr, detections, show_label=False):
    """Draw boxes on a copy of img_bgr."""
    out = img_bgr.copy()
    thick = max(2, int(img_bgr.shape[1] / 600))
    for det in detections:
        name = det["class_name"]
        conf = det["conf"]
        x1, y1, x2, y2 = det["x1"], det["y1"], det["x2"], det["y2"]
        bgr = get_color_bgr(name)
        cv2.rectangle(out, (x1, y1), (x2, y2), bgr, thick)
        if show_label:
            label = f"{name} {conf*100:.0f}%"
            font = cv2.FONT_HERSHEY_SIMPLEX
            fscale = max(0.5, img_bgr.shape[1] / 2000)
            fthick = max(1, thick - 1)
            (tw, th), _ = cv2.getTextSize(label, font, fscale, fthick)
            pad = 4
            cv2.rectangle(out, (x1, max(0, y1 - th - pad*2)), (x1 + tw + pad, y1), bgr, -1)
            cv2.putText(out, label, (x1 + pad//2, y1 - pad), font, fscale, (0,0,0), fthick, cv2.LINE_AA)
    return out


def compute_stats(values):
    if not values:
        return {"count": 0, "min": None, "max": None, "median": None, "variance": None}
    arr = np.array(values, dtype=float)
    return {
        "count": len(arr),
        "min": float(np.min(arr)),
        "max": float(np.max(arr)),
        "median": float(np.median(arr)),
        "variance": float(np.var(arr)),
    }


def print_image_summary(img_name, image_type, stats_by_class):
    tag = "🌈 COLOR (converted to gray for inference)" if image_type == "color" else "🌙 IR"
    print(f"\n{'─'*65}")
    print(f"  📷  {img_name}  [{tag}]")
    print(f"{'─'*65}")
    print(f"  {'Class':<12} {'Count':>6}  {'Min%':>6}  {'Max%':>6}  {'Med%':>6}  {'Variance':>9}")
    print(f"  {'─'*59}")
    total = 0
    for name in CLASS_ORDER:
        s = stats_by_class.get(name, compute_stats([]))
        if s["count"] > 0:
            print(f"  {name:<12} {s['count']:>6}  "
                  f"{s['min']*100:>5.1f}%  "
                  f"{s['max']*100:>5.1f}%  "
                  f"{s['median']*100:>5.1f}%  "
                  f"{s['variance']*10000:>8.2f}")
            total += s["count"]
        else:
            print(f"  {name:<12} {'—':>6}")
    print(f"  {'─'*59}")
    print(f"  {'TOTAL':<12} {total:>6}")


def show_dual_output(img_name, image_type, img_plain, img_labeled, stats_by_class):
    left = cv2.cvtColor(img_plain, cv2.COLOR_BGR2RGB)
    right = cv2.cvtColor(img_labeled, cv2.COLOR_BGR2RGB)
    tag = "COLOR → gray inference" if image_type == "color" else "IR"
    fig, axes = plt.subplots(1, 2, figsize=(22, 7), facecolor="#1a1a1a")
    for ax, img, title in zip(axes, [left, right], ["Boxes only", "Boxes + Confidence"]):
        ax.imshow(img)
        ax.set_title(title, color="white", fontsize=12, pad=8)
        ax.axis("off")
    fig.suptitle(f"{img_name}  [{tag}]", color="white", fontsize=13, y=1.01, fontweight="bold")
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, f"preview_{Path(img_name).stem}.jpg"),
                dpi=120, bbox_inches="tight", facecolor="#1a1a1a")
    plt.show()
    plt.close()
    print_image_summary(img_name, image_type, stats_by_class)


def render_final_table(total_images, color_count, ir_count, all_stats):
    """Styled summary table saved as SUMMARY_TABLE.jpg."""
    col_labels = ["Class", "Detections", "Min %", "Max %", "Median %", "Variance"]
    col_widths = [0.18, 0.14, 0.12, 0.12, 0.14, 0.14]
    col_align = ["left", "center", "center", "center", "center", "center"]
    rows = []
    row_meta = []
    grand_all = []
    for name in CLASS_ORDER:
        confs = all_stats.get(name, [])
        grand_all.extend(confs)
        s = compute_stats(confs)
        rows.append([
            name,
            str(s["count"]),
            f"{s['min']*100:.1f}%" if s["count"] else "—",
            f"{s['max']*100:.1f}%" if s["count"] else "—",
            f"{s['median']*100:.1f}%" if s["count"] else "—",
            f"{s['variance']*10000:.2f}" if s["count"] else "—",
        ])
        row_meta.append((name, CLASS_ROW_BG.get(name, "#1E1E1E")))
    gs = compute_stats(grand_all)
    rows.append([
        "ALL CLASSES",
        str(gs["count"]),
        f"{gs['min']*100:.1f}%" if gs["count"] else "—",
        f"{gs['max']*100:.1f}%" if gs["count"] else "—",
        f"{gs['median']*100:.1f}%" if gs["count"] else "—",
        f"{gs['variance']*10000:.2f}" if gs["count"] else "—",
    ])
    row_meta.append(("__total__", "#2C2C2C"))
    n_data_rows = len(rows)
    row_h = 0.11
    header_h = 0.13
    top_pad = 0.25
    bottom_pad = 0.12
    total_h = top_pad + header_h + n_data_rows * row_h + bottom_pad
    fig, ax = plt.subplots(figsize=(13, total_h * 6), facecolor="#1a1a1a")
    ax.set_facecolor("#1a1a1a")
    ax.set_xlim(0, 1)
    ax.set_ylim(0, total_h)
    ax.axis("off")
    ax.text(0.5, total_h - 0.05, "Fair Feeder V13 — Smoke Test Summary",
            ha="center", va="top", color="white", fontsize=14, fontweight="bold")
    ax.text(0.5, total_h - 0.13,
            f"Images analysed: {total_images}     🌈 Color: {color_count}     🌙 IR: {ir_count}",
            ha="center", va="top", color="#AAAAAA", fontsize=10)
    x_starts = []
    x = 0.01
    for w in col_widths:
        x_starts.append(x)
        x += w
    header_y = total_h - top_pad
    header_rect = patches.FancyBboxPatch(
        (0.01, header_y - header_h + 0.02), 0.97, header_h,
        boxstyle="round,pad=0.005", linewidth=0, facecolor="#1F4E78")
    ax.add_patch(header_rect)
    for xi, w, label, align in zip(x_starts, col_widths, col_labels, col_align):
        tx = xi + w/2 if align == "center" else xi + 0.01
        ax.text(tx, header_y - header_h/2 + 0.02, label,
                ha=align if align != "left" else "left", va="center",
                color="white", fontsize=10, fontweight="bold")
    for i, (row, (rname, bg)) in enumerate(zip(rows, row_meta)):
        y_top = header_y - header_h - i * row_h
        is_total = (rname == "__total__")
        rect = patches.FancyBboxPatch(
            (0.01, y_top - row_h + 0.005), 0.97, row_h - 0.01,
            boxstyle="round,pad=0.003", linewidth=0, facecolor=bg)
        ax.add_patch(rect)
        for j, (xi, w, val, align) in enumerate(zip(x_starts, col_widths, row, col_align)):
            tx = xi + w/2 if align == "center" else xi + 0.01
            if j == 0:
                r_c, g_c, b_c = CLASS_COLORS_RGB.get(rname, (255, 215, 0)) if not is_total else (255, 215, 0)
                txt_color = "#%02X%02X%02X" % (r_c, g_c, b_c) if not is_total else "#FFD700"
            else:
                txt_color = "#FFD700" if is_total else "white"
            ax.text(tx, y_top - row_h/2 + 0.005, val,
                    ha="left" if align == "left" else "center", va="center",
                    color=txt_color, fontsize=10,
                    fontweight="bold" if is_total else "normal")
    ax.text(0.5, 0.04,
            f"conf_threshold={CONF_THRESHOLD}  |  iou_threshold={IOU_THRESHOLD}  |  imgsz={IMGSZ}  |  Variance × 10⁴",
            ha="center", va="bottom", color="#555555", fontsize=8)
    table_path = os.path.join(OUTPUT_DIR, "SUMMARY_TABLE.jpg")
    plt.savefig(table_path, dpi=150, bbox_inches="tight", facecolor="#1a1a1a")
    plt.show()
    plt.close()
    print(f"\n✅  Summary table saved → {table_path}")


print("✅ Helper functions defined.")

In [ ]:
# ───────────────────────────────────────────────────────────────
# Feeding Behaviour Tracker
# ───────────────────────────────────────────────────────────────

class FeedingTracker:
    """Tracks feeding events across video frames.

    Eating logic (both combined):
      Primary signal:  Cat bbox overlaps Bowl bbox (IoU > threshold)
      Confirmation:    Kibble bbox count decreases during the session
    """

    def __init__(self, fps=25.0):
        self.fps = fps

        # Per-frame arrays
        self.kibble_counts = []       # int per frame
        self.dan_at_bowl = []         # bool per frame
        self.sanbo_at_bowl = []       # bool per frame
        self.dan_hand_present = []    # bool per frame
        self.timestamps = []          # OCR string per frame

        # Events
        self.dan_first_frame = None
        self.dan_first_timestamp = None
        self.sanbo_first_frame = None
        self.sanbo_first_timestamp = None
        self.snapshots = {}  # {"sanbo_arrival": img, "dan_hand_ep0": img, ...}

    def process_frame(self, frame_idx, detections, timestamp_str, frame_img):
        """Process one frame's detections."""
        # Kibble count
        kibble_count = sum(1 for d in detections if d["class_name"] == "Kibble")
        self.kibble_counts.append(kibble_count)
        self.timestamps.append(timestamp_str)

        # Find Bowl bbox (use largest if multiple)
        bowl_boxes = [d for d in detections if d["class_name"] == "Bowl"]
        bowl_box = max(bowl_boxes, key=lambda d: (d["x2"]-d["x1"])*(d["y2"]-d["y1"])) if bowl_boxes else None

        # Cat-at-bowl overlap
        dan_at = False
        sanbo_at = False
        if bowl_box:
            for d in detections:
                if d["class_name"] == "Dan" and bbox_iou(d, bowl_box) > OVERLAP_IOU_THRESHOLD:
                    dan_at = True
                elif d["class_name"] == "Sanbo" and bbox_iou(d, bowl_box) > OVERLAP_IOU_THRESHOLD:
                    sanbo_at = True
        self.dan_at_bowl.append(dan_at)
        self.sanbo_at_bowl.append(sanbo_at)

        # Dan_hand
        dan_hand = any(d["class_name"] == "Dan_hand" for d in detections)
        self.dan_hand_present.append(dan_hand)

        # First appearances + snapshots
        dan_here = any(d["class_name"] == "Dan" for d in detections)
        if dan_here and self.dan_first_frame is None:
            self.dan_first_frame = frame_idx
            self.dan_first_timestamp = timestamp_str

        sanbo_here = any(d["class_name"] == "Sanbo" for d in detections)
        if sanbo_here and self.sanbo_first_frame is None:
            self.sanbo_first_frame = frame_idx
            self.sanbo_first_timestamp = timestamp_str
            annotated_snap = draw_boxes(frame_img, detections, show_label=True)
            self.snapshots["sanbo_arrival"] = annotated_snap

        # Snapshot for first Dan_hand frame in each episode
        if dan_hand:
            # Check if this is start of a new episode
            prev_hand = self.dan_hand_present[-2] if len(self.dan_hand_present) >= 2 else False
            if not prev_hand:
                ep_idx = self._count_hand_episodes_so_far()
                annotated_snap = draw_boxes(frame_img, detections, show_label=True)
                self.snapshots[f"dan_hand_ep{ep_idx}"] = annotated_snap

    def _count_hand_episodes_so_far(self):
        """Count how many Dan_hand episodes have started so far."""
        episodes = 0
        in_episode = False
        gap = 0
        for present in self.dan_hand_present:
            if present:
                if not in_episode:
                    episodes += 1
                    in_episode = True
                gap = 0
            else:
                gap += 1
                if gap >= DAN_HAND_GAP_FRAMES:
                    in_episode = False
        return episodes - 1  # current one is starting, so index = count-1

    def _find_episodes(self, bool_array, gap_frames):
        """Find contiguous True episodes with a minimum gap between them.
        Returns list of (start_frame, end_frame)."""
        episodes = []
        start = None
        gap = 0
        for i, val in enumerate(bool_array):
            if val:
                if start is None:
                    start = i
                gap = 0
            else:
                gap += 1
                if start is not None and gap >= gap_frames:
                    episodes.append((start, i - gap))
                    start = None
        if start is not None:
            episodes.append((start, len(bool_array) - 1))
        return episodes

    def _get_stable_kibble_count(self, around_frame, window=5, direction="before"):
        """Get median kibble count in a window around a frame,
        excluding frames where cat/hand is at bowl (occlusion)."""
        if direction == "before":
            start = max(0, around_frame - window)
            end = around_frame
        else:
            start = around_frame
            end = min(len(self.kibble_counts), around_frame + window)
        counts = []
        for i in range(start, end):
            if i < len(self.kibble_counts):
                # Skip frames where cat or hand may be occluding
                if not self.dan_at_bowl[i] and not self.sanbo_at_bowl[i] and not self.dan_hand_present[i]:
                    counts.append(self.kibble_counts[i])
        return int(np.median(counts)) if counts else (self.kibble_counts[around_frame] if around_frame < len(self.kibble_counts) else 0)

    def _get_timestamp(self, frame_idx):
        """Get the closest non-empty timestamp for a frame."""
        if frame_idx < len(self.timestamps) and self.timestamps[frame_idx]:
            return self.timestamps[frame_idx]
        # Search nearby frames
        for offset in range(1, 60):
            for idx in [frame_idx - offset, frame_idx + offset]:
                if 0 <= idx < len(self.timestamps) and self.timestamps[idx]:
                    return self.timestamps[idx]
        return "??"

    def summarize(self):
        """Compute feeding summary after all frames are processed."""
        n_frames = len(self.kibble_counts)
        if n_frames == 0:
            return {"error": "No frames processed"}

        # Time range
        start_ts = self._get_timestamp(0)
        end_ts = self._get_timestamp(n_frames - 1)

        # Dan_hand episodes
        hand_episodes = self._find_episodes(self.dan_hand_present, DAN_HAND_GAP_FRAMES)
        hand_summary = []
        for start_f, end_f in hand_episodes:
            kibble_before = self._get_stable_kibble_count(start_f, window=10, direction="before")
            kibble_after = self._get_stable_kibble_count(end_f, window=10, direction="after")
            kibble_added = max(0, kibble_after - kibble_before)
            hand_summary.append({
                "start_frame": start_f,
                "end_frame": end_f,
                "timestamp": self._get_timestamp(start_f),
                "kibble_added": kibble_added,
            })

        # Dan eating sessions
        dan_bowl_episodes = self._find_episodes(self.dan_at_bowl, gap_frames=10)
        dan_kibble_eaten = 0
        dan_bowl_seconds = 0
        for start_f, end_f in dan_bowl_episodes:
            kibble_before = self._get_stable_kibble_count(start_f, window=10, direction="before")
            kibble_after = self._get_stable_kibble_count(end_f, window=10, direction="after")
            drop = kibble_before - kibble_after
            if drop >= EATING_KIBBLE_DROP:
                dan_kibble_eaten += drop
            dan_bowl_seconds += (end_f - start_f) / self.fps

        # Sanbo eating sessions
        sanbo_bowl_episodes = self._find_episodes(self.sanbo_at_bowl, gap_frames=10)
        sanbo_kibble_eaten = 0
        sanbo_bowl_seconds = 0
        for start_f, end_f in sanbo_bowl_episodes:
            kibble_before = self._get_stable_kibble_count(start_f, window=10, direction="before")
            kibble_after = self._get_stable_kibble_count(end_f, window=10, direction="after")
            drop = kibble_before - kibble_after
            if drop >= EATING_KIBBLE_DROP:
                sanbo_kibble_eaten += drop
            sanbo_bowl_seconds += (end_f - start_f) / self.fps

        return {
            "n_frames": n_frames,
            "duration_sec": n_frames / self.fps,
            "start_ts": start_ts,
            "end_ts": end_ts,
            "dan_first_ts": self.dan_first_timestamp,
            "sanbo_first_ts": self.sanbo_first_timestamp,
            "hand_episodes": hand_summary,
            "dan_kibble_eaten": dan_kibble_eaten,
            "dan_bowl_seconds": dan_bowl_seconds,
            "sanbo_kibble_eaten": sanbo_kibble_eaten,
            "sanbo_bowl_seconds": sanbo_bowl_seconds,
        }


def format_duration(seconds):
    m, s = divmod(int(seconds), 60)
    return f"{m}m {s:02d}s"


def format_feeding_summary(summary, video_name):
    """Format a summary dict into a readable string (for print + Discord)."""
    lines = []
    lines.append(f"\U0001f431 Fair Feeder Summary \u2014 {video_name}")
    lines.append(f"   {summary['start_ts']} \u2192 {summary['end_ts']}")
    lines.append(f"   Duration: {format_duration(summary['duration_sec'])}")
    lines.append("\u2501" * 48)

    # Dan_hand feeding
    hand_eps = summary["hand_episodes"]
    lines.append(f"\n\U0001f4e6 Feeding (Dan_hand): {len(hand_eps)} attempt(s)")
    for ep in hand_eps:
        lines.append(f"   {ep['timestamp']} \u2014 ~{ep['kibble_added']} kibble added")

    # Cat arrivals
    if summary["sanbo_first_ts"]:
        lines.append(f"\n\U0001f408 Sanbo arrived at {summary['sanbo_first_ts']}")
    else:
        lines.append("\n\U0001f408 Sanbo: not detected")
    if summary["dan_first_ts"]:
        lines.append(f"\U0001f408\u200d\u2b1b Dan present from {summary['dan_first_ts']}")
    else:
        lines.append("\U0001f408\u200d\u2b1b Dan: not detected")

    # Eating estimates
    lines.append(f"\n\U0001f37d\ufe0f Eating estimate:")
    lines.append(f"   Dan:   ~{summary['dan_kibble_eaten']} kibble (at bowl {format_duration(summary['dan_bowl_seconds'])})")
    lines.append(f"   Sanbo: ~{summary['sanbo_kibble_eaten']} kibble (at bowl {format_duration(summary['sanbo_bowl_seconds'])})")

    # Dan_hand attempts
    lines.append(f"\n\u270b Dan's hand attempts: {len(hand_eps)}")

    return "\n".join(lines)


def plot_video_timeline(tracker, video_name):
    """Plot timeline chart: kibble count, cat presence, Dan_hand events."""
    n = len(tracker.kibble_counts)
    if n == 0:
        return
    frames = np.arange(n)
    time_sec = frames / tracker.fps

    fig, axes = plt.subplots(4, 1, figsize=(16, 8), sharex=True, facecolor="#1a1a1a")
    fig.suptitle(f"Detection Timeline: {video_name}", color="white", fontsize=13, fontweight="bold")

    # Kibble count
    axes[0].fill_between(time_sec, tracker.kibble_counts, color="#FCF400", alpha=0.7)
    axes[0].set_ylabel("Kibble", color="white", fontsize=10)
    axes[0].set_facecolor("#1a1a1a")
    axes[0].tick_params(colors="white")

    # Dan at bowl
    dan_arr = np.array(tracker.dan_at_bowl, dtype=float)
    axes[1].fill_between(time_sec, dan_arr, color="#00FFCE", alpha=0.7, step="mid")
    axes[1].set_ylabel("Dan\nat bowl", color="white", fontsize=10)
    axes[1].set_ylim(-0.1, 1.1)
    axes[1].set_facecolor("#1a1a1a")
    axes[1].tick_params(colors="white")

    # Sanbo at bowl
    sanbo_arr = np.array(tracker.sanbo_at_bowl, dtype=float)
    axes[2].fill_between(time_sec, sanbo_arr, color="#FF8000", alpha=0.7, step="mid")
    axes[2].set_ylabel("Sanbo\nat bowl", color="white", fontsize=10)
    axes[2].set_ylim(-0.1, 1.1)
    axes[2].set_facecolor("#1a1a1a")
    axes[2].tick_params(colors="white")

    # Dan_hand
    hand_arr = np.array(tracker.dan_hand_present, dtype=float)
    axes[3].fill_between(time_sec, hand_arr, color="#0010FF", alpha=0.7, step="mid")
    axes[3].set_ylabel("Dan\nhand", color="white", fontsize=10)
    axes[3].set_ylim(-0.1, 1.1)
    axes[3].set_xlabel("Time (seconds)", color="white", fontsize=10)
    axes[3].set_facecolor("#1a1a1a")
    axes[3].tick_params(colors="white")

    plt.tight_layout()
    timeline_path = os.path.join(OUTPUT_DIR, f"timeline_{Path(video_name).stem}.jpg")
    plt.savefig(timeline_path, dpi=120, bbox_inches="tight", facecolor="#1a1a1a")
    plt.show()
    plt.close()
    print(f"\u2705 Timeline saved \u2192 {timeline_path}")


print("✅ FeedingTracker defined.")

In [ ]:
# ───────────────────────────────────────────────────────────────
# Scan SOURCE_DIR + Process Images
# ───────────────────────────────────────────────────────────────

all_files = sorted(Path(SOURCE_DIR).iterdir())
image_paths = [f for f in all_files if classify_file(f) == "image"]
video_paths = [f for f in all_files if classify_file(f) == "video"]

print(f"✅ Found {len(image_paths)} image(s) and {len(video_paths)} video(s) in SOURCE_DIR")

if not image_paths:
    print(f"   No images found in: {SOURCE_DIR}")
else:
    print(f"\n{'='*65}")
    print(f"  Processing {len(image_paths)} image(s)...")
    print(f"{'='*65}")

    all_stats = {}
    color_count = 0
    ir_count = 0

    for img_path in image_paths:
        img_name = img_path.name
        img_bgr = cv2.imread(str(img_path))
        if img_bgr is None:
            print(f"\u26a0\ufe0f  Could not read: {img_name} \u2014 skipping")
            continue

        image_type = detect_image_type(img_bgr)
        if image_type == "color":
            color_count += 1
        else:
            ir_count += 1

        inference_img = prepare_for_inference(img_bgr, image_type)
        tmp_path = "/tmp/_infer_tmp.jpg"
        cv2.imwrite(tmp_path, inference_img)

        results = model.predict(
            source=tmp_path,
            imgsz=IMGSZ,
            conf=CONF_THRESHOLD,
            iou=IOU_THRESHOLD,
            rect=True,
            verbose=False,
        )

        detections = parse_results(results)
        confs_by_class = {}
        for det in detections:
            confs_by_class.setdefault(det["class_name"], []).append(det["conf"])

        stats_by_class = {name: compute_stats(confs) for name, confs in confs_by_class.items()}
        for name, confs in confs_by_class.items():
            all_stats.setdefault(name, []).extend(confs)

        img_plain = draw_boxes(img_bgr, detections, show_label=False)
        img_labeled = draw_boxes(img_bgr, detections, show_label=True)

        stem = Path(img_name).stem
        cv2.imwrite(os.path.join(OUTPUT_DIR, f"{stem}_boxes_only.jpg"), img_plain)
        cv2.imwrite(os.path.join(OUTPUT_DIR, f"{stem}_with_conf.jpg"), img_labeled)

        show_dual_output(img_name, image_type, img_plain, img_labeled, stats_by_class)

    # Final summary table for all images
    if all_stats:
        render_final_table(
            total_images=len(image_paths),
            color_count=color_count,
            ir_count=ir_count,
            all_stats=all_stats,
        )

    print(f"\n✅  Image outputs saved to: {OUTPUT_DIR}")

In [ ]:
# ───────────────────────────────────────────────────────────────
# Process Videos
# ───────────────────────────────────────────────────────────────

# Re-scan SOURCE_DIR so this cell works even if run independently
all_files = sorted(Path(SOURCE_DIR).iterdir())
video_paths = [f for f in all_files if classify_file(f) == "video"]
print(f"✅ Found {len(video_paths)} video(s) in SOURCE_DIR")

video_summaries = []   # collect for Discord notification later

if not video_paths:
    print(f"   No videos found in: {SOURCE_DIR}")
else:
    for vid_path in video_paths:
        vid_name = vid_path.name
        print(f"\n{'='*65}")
        print(f"  Processing video: {vid_name}")
        print(f"{'='*65}")

        cap = cv2.VideoCapture(str(vid_path))
        if not cap.isOpened():
            print(f"\u26a0\ufe0f  Could not open: {vid_name} \u2014 skipping")
            continue

        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
        width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        print(f"   {width}x{height} @ {fps:.1f} fps, {total_frames} frames")
        print(f"   Estimated time: ~{total_frames / 2.5 / 60:.1f} min (at ~2.5 FPS on CPU)")

        # Output video writer
        out_name = f"{vid_path.stem}_annotated.mp4"
        out_path = os.path.join(OUTPUT_DIR, out_name)
        fourcc = cv2.VideoWriter_fourcc(*'mp4v')
        writer = cv2.VideoWriter(out_path, fourcc, fps, (width, height))

        tracker = FeedingTracker(fps=fps)
        last_ts = ""

        for frame_idx in tqdm(range(total_frames), desc=vid_name, unit="frame"):
            ret, frame = cap.read()
            if not ret:
                break

            # OCR timestamp every Nth frame
            if frame_idx % OCR_EVERY_N_FRAMES == 0:
                try:
                    last_ts = extract_timestamp(frame)
                except Exception:
                    pass  # keep last_ts

            # Prepare for inference (IR detection + conversion)
            image_type = detect_image_type(frame)
            inference_img = prepare_for_inference(frame, image_type)

            # YOLO inference
            results = model.predict(
                source=inference_img,
                imgsz=IMGSZ,
                conf=CONF_THRESHOLD,
                iou=IOU_THRESHOLD,
                verbose=False,
            )

            # Parse detections
            detections = parse_results(results)

            # Draw on original frame and write to output video
            annotated = draw_boxes(frame, detections, show_label=True)
            writer.write(annotated)

            # Feed to tracker
            tracker.process_frame(frame_idx, detections, last_ts, frame)

        cap.release()
        writer.release()
        print(f"\n\u2705 Annotated video saved \u2192 {out_path}")

        # --- Summary ---
        summary = tracker.summarize()
        summary_text = format_feeding_summary(summary, vid_name)
        print(f"\n{summary_text}")

        # --- Save snapshots ---
        snapshot_paths = []
        for event_name, snap_img in tracker.snapshots.items():
            snap_path = os.path.join(OUTPUT_DIR, f"{vid_path.stem}_{event_name}.jpg")
            cv2.imwrite(snap_path, snap_img)
            snapshot_paths.append(snap_path)
            print(f"\u2705 Snapshot saved \u2192 {snap_path}")

        # --- Timeline chart ---
        plot_video_timeline(tracker, vid_name)

        # Collect for Discord
        video_summaries.append({
            "video_name": vid_name,
            "summary_text": summary_text,
            "snapshot_paths": snapshot_paths,
        })

        # --- Try to display video inline ---
        try:
            display(Video(out_path, embed=True, width=800))
        except Exception:
            print("   (Video display not supported inline; download from Google Drive)")

    print(f"\n\u2705  All video outputs saved to: {OUTPUT_DIR}")

In [ ]:
# ───────────────────────────────────────────────────────────────
# Discord Notification
# ───────────────────────────────────────────────────────────────

def send_discord_summary(webhook_url, summary_text, snapshot_paths):
    """Send feeding summary + snapshots to Discord via webhook."""
    if not webhook_url:
        print("\u26a0\ufe0f  No Discord webhook URL configured. Skipping notification.")
        return False

    # Discord has a 2000 char limit per message
    # Truncate if needed, but our summaries should be short enough
    content = summary_text[:1900]

    # Build multipart form data with text + images
    payload = {"content": content}
    files_data = {}
    file_handles = []

    for i, path in enumerate(snapshot_paths[:10]):  # Discord max 10 files
        fh = open(path, "rb")
        file_handles.append(fh)
        files_data[f"file{i}"] = (Path(path).name, fh, "image/jpeg")

    try:
        resp = requests.post(
            webhook_url,
            data={"payload_json": json.dumps(payload)},
            files=files_data,
        )
        if resp.status_code in (200, 204):
            print("\u2705 Discord notification sent!")
            return True
        else:
            print(f"\u274c Discord error: {resp.status_code} \u2014 {resp.text[:200]}")
            return False
    finally:
        for fh in file_handles:
            fh.close()


# --- Send all video summaries ---
if video_summaries:
    print(f"\nSending {len(video_summaries)} summary(ies) to Discord...\n")
    for vs in video_summaries:
        send_discord_summary(
            DISCORD_WEBHOOK_URL,
            vs["summary_text"],
            vs["snapshot_paths"],
        )
else:
    print("No video summaries to send.")

## Results

All annotated outputs are saved to the output directory on Google Drive.

- **Images**: `{filename}_boxes_only.jpg` + `{filename}_with_conf.jpg`
- **Videos**: `{filename}_annotated.mp4`
- **Snapshots**: `{filename}_sanbo_arrival.jpg`, `{filename}_dan_hand_ep0.jpg`, etc.
- **Charts**: `timeline_{filename}.jpg`, `SUMMARY_TABLE.jpg`

The feeding summary is printed above and sent to Discord (if webhook URL is configured).

### Summary format
```
🐱 Fair Feeder Summary — video.mp4
   2024-01-15 18:30:00 → 19:00:00
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

📦 Feeding (Dan_hand): 2 attempt(s)
   18:32:15 — ~5 kibble added
   18:45:30 — ~3 kibble added

🐈 Sanbo arrived at 18:35:42
🐈‍⬛ Dan present from 18:30:05

🍽️ Eating estimate:
   Dan:   ~12 kibble (at bowl 5m 30s)
   Sanbo: ~8 kibble  (at bowl 3m 15s)

✋ Dan's hand attempts: 2
```